# M and A, workshop 3 — Loading to SQL, and proving it worked

You have a merged customer table. The last step of the migration is to put it in the database —
and then the step nobody teaches: **showing that what came out is what went in.**

A migration that loses 4 % of its rows looks exactly like one that loses none, right up until a
colleague asks why the customer count dropped. Today you build the checks that answer that
question before anyone asks it.

1. **CSV to SQLite**, with the types you meant.
2. **Reconciliation**: count, nulls, keys, round-trip.
3. **Break it on purpose**, and watch a check catch it.

In [ ]:
# ABOUTME: M and A workshop 3 - loading the merged customers into SQLite and reconciling the load.
# ABOUTME: Row counts, null rates, key uniqueness and a round-trip diff.
import io
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 150)
DB = Path("customers.db")
if DB.exists():
    DB.unlink()          # a workshop should start from nothing every time it is run

## The merged table

Pretend last week went well: here are 10 cleaned customers in the target schema, as a **CSV
string** — which is exactly what your `my_m_and_a` function returns, and what `csv_to_sql` from
My DS Babel expects.

Note what a realistic result looks like: some ages are missing, half the countries are unknown,
and `created_at` and `referral` are empty for everyone because no source system had them.

In [ ]:
merged_csv = """gender,firstname,lastname,email,age,city,country,created_at,referral
Female,Latesha,Barton,latesha.barton@yahoo.com,85,Indianapolis,United States of America,,
Female,Yolando,Jast,yolando_jast@yahoo.com,53,Columbus,,,
Male,Brandon,Hyatt,brandon.hyatt@gmail.com,41,New York,United States of America,,
Male,Chase,Oberbrunner,chase.oberbrunner@hotmail.com,29,Austin,United States of America,,
Male,Erwin,Kuhic,erwin.kuhic@yahoo.com,,Phoenix,United States of America,,
Female,Tawanda,Lind,tawanda.lind@yahoo.com,37,Seattle,United States of America,,
Female,Lorette,Kutch,kutch.lorette@gmail.com,51,Austin,,,
Female,Clorinda,Wiza,clorinda.wiza@gmail.com,84,San Diego,,,
Male,Joesph,Littel,littel.joesph@gmail.com,97,Columbus,,,
Male,Carlo,Stokes,carlo.stokes@hotmail.com,,New York,,,
"""

merged = pd.read_csv(io.StringIO(merged_csv))
print(merged.shape)
merged

(10, 9)


,gender,firstname,lastname,email,age,city,country,created_at,referral
0,Female,Latesha,Barton,latesha.barton@yahoo.com,85.0,Indianapolis,United States of America,NaN,NaN
1,Female,Yolando,Jast,yolando_jast@yahoo.com,53.0,Columbus,NaN,NaN,NaN
2,Male,Brandon,Hyatt,brandon.hyatt@gmail.com,41.0,New York,United States of America,NaN,NaN
3,Male,Chase,Oberbrunner,chase.oberbrunner@hotmail.com,29.0,Austin,United States of America,NaN,NaN
4,Male,Erwin,Kuhic,erwin.kuhic@yahoo.com,NaN,Phoenix,United States of America,NaN,NaN
5,Female,Tawanda,Lind,tawanda.lind@yahoo.com,37.0,Seattle,United States of America,NaN,NaN
6,Female,Lorette,Kutch,kutch.lorette@gmail.com,51.0,Austin,NaN,NaN,NaN
7,Female,Clorinda,Wiza,clorinda.wiza@gmail.com,84.0,San Diego,NaN,NaN,NaN
8,Male,Joesph,Littel,littel.joesph@gmail.com,97.0,Columbus,NaN,NaN,NaN
9,Male,Carlo,Stokes,carlo.stokes@hotmail.com,NaN,New York,NaN,NaN,NaN


## 1. Getting it into SQLite

`sqlite3` ships with Python; there is no server to install. Three lines open a database, run SQL,
and close it.

pandas can do the whole load in one call — `DataFrame.to_sql` — and that is what you would use at
work. Your Qwasar `csv_to_sql` does it by hand with `INSERT` statements, which is worth doing once
because it forces you to see the column list, the placeholders and the commit.

**Rule of thumb:** always `commit()`. A SQLite write that is never committed is a write that never
happened, and it fails silently.

In [ ]:
with sqlite3.connect(DB) as conn:
    merged.to_sql("customers", conn, index=False, if_exists="replace")

with sqlite3.connect(DB) as conn:
    print(pd.read_sql("SELECT COUNT(*) AS rows FROM customers", conn).to_string(index=False))
    print()
    print(pd.read_sql("PRAGMA table_info(customers)", conn)[["name", "type"]].to_string(index=False))

 rows
   10

      name type
    gender TEXT
 firstname TEXT
  lastname TEXT
     email TEXT
       age REAL
      city TEXT
   country TEXT
created_at REAL
  referral REAL


Look at the `type` column. pandas inferred `TEXT` for the strings and `REAL` for `age` — not
`INTEGER`, because the two missing ages forced the column to float. Nobody is 37.0 years old.

This is the first thing a reconciliation check would catch, and it is a one-line fix at load time.

### Exercise 1 — load with the types you meant

Write `load_customers(df, db_path)`: create the table with an **explicit schema** (ages as
`INTEGER`, everything else `TEXT`, email `NOT NULL`), then insert the rows.

The `CREATE TABLE` is done for you; the exercise is the insert. A SQLite insert takes one `?` per
column and a sequence of row tuples: `conn.executemany("INSERT INTO customers VALUES (?,?,...)", rows)`.

In [ ]:
SCHEMA = """
CREATE TABLE customers (
    gender     TEXT,
    firstname  TEXT,
    lastname   TEXT,
    email      TEXT NOT NULL,
    age        INTEGER,
    city       TEXT,
    country    TEXT,
    created_at TEXT,
    referral   TEXT
)
"""

def load_customers(df, db_path):
    """Create the customers table with an explicit schema and insert every row."""
    if Path(db_path).exists():
        Path(db_path).unlink()
    placeholders = ",".join("?" * df.shape[1])
    rows = list(df.astype(object).where(df.notna(), None).itertuples(index=False, name=None))
    with sqlite3.connect(db_path) as conn:
        conn.execute(SCHEMA)
        conn.executemany(f"INSERT INTO customers VALUES ({placeholders})", rows)
        conn.commit()
    return db_path

load_customers(merged, "customers_typed.db")
with sqlite3.connect("customers_typed.db") as conn:
    print(pd.read_sql("PRAGMA table_info(customers)", conn)[["name", "type"]].to_string(index=False))

      name    type
    gender    TEXT
 firstname    TEXT
  lastname    TEXT
     email    TEXT
       age INTEGER
      city    TEXT
   country    TEXT
created_at    TEXT
  referral    TEXT


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

def _query(sql, db="customers_typed.db"):
    with sqlite3.connect(db) as conn:
        return pd.read_sql(sql, conn)

check("the table exists", lambda: len(_query("SELECT * FROM customers")) > 0)
check("every row arrived", lambda: _query("SELECT COUNT(*) n FROM customers").n.item() == len(merged))
check("age is an INTEGER column",
      lambda: _query("PRAGMA table_info(customers)").set_index("name").type.get("age") == "INTEGER")
check("missing ages stayed missing",
      lambda: _query("SELECT COUNT(*) n FROM customers WHERE age IS NULL").n.item() == int(merged.age.isna().sum()))

  the table exists: PASS
  every row arrived: PASS
  age is an INTEGER column: PASS
  missing ages stayed missing: PASS


True

## 2. Reconciliation: four questions

The load reported no error. That is not evidence. Ask the database what it actually holds, and
compare against what you sent:

| check | what it catches |
|---|---|
| **row count** | rows silently dropped by a failed insert or a filter |
| **null rate per column** | a column that arrived empty because of a name mismatch |
| **key uniqueness** | duplicates created by loading twice |
| **round trip** | values mangled in transit (types, encodings, truncation) |

### Exercise 2 — write the checks

Write `reconcile(df, db_path)` returning a DataFrame with one row per check and a `passed` column.
Three checks minimum: row count, distinct email count, and per-column null counts matching.

In [ ]:
def reconcile(df, db_path):
    """Compare the source DataFrame against what the database actually stored."""
    with sqlite3.connect(db_path) as conn:
        loaded = pd.read_sql("SELECT * FROM customers", conn)

    source_nulls = df.isna().sum().sum()
    loaded_nulls = loaded.isna().sum().sum()
    checks = [
        {"check": "row count", "source": len(df), "loaded": len(loaded),
         "passed": len(df) == len(loaded)},
        {"check": "distinct emails", "source": df.email.nunique(), "loaded": loaded.email.nunique(),
         "passed": df.email.nunique() == loaded.email.nunique()},
        {"check": "total nulls", "source": source_nulls, "loaded": loaded_nulls,
         "passed": source_nulls == loaded_nulls},
        {"check": "columns", "source": list(df.columns) == list(loaded.columns),
         "loaded": list(loaded.columns) == list(df.columns),
         "passed": list(df.columns) == list(loaded.columns)},
    ]
    return pd.DataFrame(checks)

reconcile(merged, "customers_typed.db")

,check,source,loaded,passed
0,row count,10,10,True
1,distinct emails,10,10,True
2,total nulls,27,27,True
3,columns,True,True,True


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

_rec = reconcile(merged, "customers_typed.db")
check("returns at least three checks", lambda: len(_rec) >= 3)
check("has a passed column", lambda: "passed" in _rec.columns)
check("a correct load passes everything", lambda: bool(_rec.passed.all()))

  returns at least three checks: PASS
  has a passed column: PASS
  a correct load passes everything: PASS


True

### Exercise 3 — break it, and catch it

A check you have never seen fail is a check you do not know works. Load the table again **without
clearing it first** — the classic mistake of re-running a migration script — and confirm your
reconciliation notices.

In [ ]:
placeholders = ",".join("?" * merged.shape[1])
rows = list(merged.astype(object).where(merged.notna(), None).itertuples(index=False, name=None))
with sqlite3.connect("customers_typed.db") as conn:
    conn.executemany(f"INSERT INTO customers VALUES ({placeholders})", rows)
    conn.commit()

broken = reconcile(merged, "customers_typed.db")
broken

,check,source,loaded,passed
0,row count,10,20,False
1,distinct emails,10,10,True
2,total nulls,27,54,False
3,columns,True,True,True


Two checks go red and they tell different stories: the row count doubled, but the distinct-email
count did not move. That pair is the signature of **a duplicate load**, not of extra customers —
and it is precisely why counting rows alone is not enough.

**Rule of thumb:** make your load repeatable. Dropping and recreating the table, or an
`INSERT ... ON CONFLICT` on a unique email, means running the script twice is harmless.

### Exercise 4 — the round trip

The strongest check: pull everything back out of the database, and compare it to what you sent,
value by value. If `sql_to_csv` and `csv_to_sql` are inverses, this is empty.

In [ ]:
# Start from a clean, correct load again.
load_customers(merged, "customers_typed.db")

def round_trip_diff(df, db_path):
    """Return the rows that differ between the source frame and the database contents."""
    with sqlite3.connect(db_path) as conn:
        loaded = pd.read_sql("SELECT * FROM customers", conn)

    def canonical(frame):
        # Compare as text: SQLite hands back 85 where pandas held 85.0, and that is not a difference.
        out = frame[list(df.columns)].astype(object).where(frame.notna(), "")
        for col in out.columns:
            out[col] = out[col].map(lambda v: str(int(v)) if isinstance(v, float) and v.is_integer() else str(v))
        return out.sort_values(list(df.columns)).reset_index(drop=True)

    a, b = canonical(df), canonical(loaded)
    unequal = (a != b).any(axis=1)
    return a[unequal].join(b[unequal], rsuffix="_db")

diff = round_trip_diff(merged, "customers_typed.db")
print(f"{len(diff)} differing rows")
diff

0 differing rows


,gender,firstname,lastname,email,age,city,country,created_at,referral,gender_db,firstname_db,lastname_db,email_db,age_db,city_db,country_db,created_at_db,referral_db


In [ ]:
def check(label, *conditions):
    """Report an exercise self-check without raising: a red traceback is a bad first impression."""
    try:
        ok = all(bool(c()) for c in conditions)
    except Exception as exc:                      # a stub that still returns None lands here
        print(f"  {label}: not yet ({type(exc).__name__}: {exc})")
        return False
    print(f"  {label}: {'PASS' if ok else 'not yet'}")
    return ok

check("the round trip is clean", lambda: len(round_trip_diff(merged, "customers_typed.db")) == 0)

  the round trip is clean: PASS


True

Zero differing rows. *Now* you can tell your manager the migration is done, and show them why.

## 3. One more thing: do not commit the data

The project brief is blunt about it — if the acquired company's CSVs are found in your repository
at the peer review, the project fails. The same rule applies at work for a less dramatic reason:
customer data does not belong in version control, where it is copied to every laptop that clones
the repo and lives forever in the history.

```
# .gitignore
data/
*.db
*.csv
```

Your code must therefore **fetch or receive** its input rather than carry it. That is also why
`my_m_and_a` takes CSV *contents* as arguments instead of filenames.

## What to take away

1. **Declare your schema.** Letting the loader guess gives you ages of 37.0 and dates stored as text.
2. **A load without an error is not a verified load.** Count rows, count nulls, count keys.
3. **Row count alone is not enough** — rows doubled while distinct emails stayed flat is a
   duplicate load, and only the pair of checks tells you that.
4. **Make loads repeatable**, so running the script twice is not an incident.
5. **Round-trip when you can.** Out and back, compared as text, is the closest thing to proof.
6. **Never commit the customer data.**

### Take it home
Run your real `my_m_and_a` output through `csv_to_sql`, then write the four checks above against
it. Bring the reconciliation table to the next session — if a check fails, bring that too, it is
the more interesting case.